# L5-B: Packing 2-bit Weights

In this lesson, you will learn how to store low precision weights through a technique called "packing".

## Packing

In [2]:
import torch

**Note:** Younes will explain the below code, and walk through each iteration step. You can go through the comprehensive explanation in the markdown below after first watching Younes's explanation.

```Python
# Example Tensor: [1, 0, 3, 2]
    # 1 0 3 2 - 01 00 11 10

    # Starting point of packed int8 Tensor
    # [0000 0000]
    
    ##### First Iteration Start:
    # packed int8 Tensor State: [0000 0000]
    # 1 = 0000 0001
    # 0000 0001
    # No left shifts in the First Iteration
    # After bit-wise OR operation between 0000 0000 and 0000 0001:
    # packed int8 Tensor State: 0000 0001
    ##### First Iteration End

    ##### Second Iteration Start:
    # packed int8 Tensor State: [0000 0001]
    # 0 = 0000 0000
    # 0000 0000
    # 2 left shifts:
    # [0000 0000] (1 shift)-> 0000 0000 (2 shift)-> 0000 0000
    # After bit-wise OR operation between 0000 0001 and 0000 0000:
    # packed int8 Tensor State: 0000 0001
    ##### Second Iteration End

    ##### Third Iteration Start:
    # packed int8 Tensor State: [0000 0001]
    # 3 = 0000 0011
    # 0000 0011
    # 4 left shifts:
    # [0000 0011] (1 shift)-> 0000 0110 (2 shift)-> 0000 1100
    # 0000 1100 (3 shift)-> 0001 1000 (4 shift)-> 0011 0000
    # After bit-wise OR operation between 0000 0001 and 0011 0000:
    # packed int8 Tensor State: 0011 0001
    ##### Third Iteration End

    ##### Fourth Iteration Start:
    # packed int8 Tensor State: [0011 0001]
    # 2 = 0000 0010
    # 0000 0010
    # 6 left shifts:
    # [0000 0010] (1 shift)-> 0000 0100 (2 shift)-> 0000 1000
    # 0000 1000 (3 shift)-> 0001 0000 (4 shift)-> 0010 0000
    # 0010 0000 (5 shift)-> 0100 0000 (6 shift)-> 1000 0000
    # After bit-wise OR operation between 0011 0001 and 1000 0000:
    # packed int8 Tensor State: 1011 0001
    ##### Fourth Iteration End
    
    # Final packed int8 Tensor State: [1011 0001]
```

In [3]:
def pack_weights(uint8tensor, bits):
    if uint8tensor.shape[0] * bits % 8 != 0:
        raise ValueError(f"The input shape needs to be a mutiple of {8 / bits} - got {uint8tensor.shape[0]}")

    num_values = uint8tensor.shape[0] * bits // 8

    num_steps = 8 // bits

    unpacked_idx = 0

    packed_tensor = torch.zeros((num_values), dtype=torch.uint8)

    # 1 0 3 2 - 01 00 11 10

    # [0000 0000] -> 0000 0001

    # 0000 0001

    # 0000 0000 - 0000 0000

    # 0000 0011 - 0011 0000 - 0011 0001

    # 1011 0001

    for i in range(num_values):
        for j in range(num_steps):
            packed_tensor[i] |= uint8tensor[unpacked_idx] << (bits * j)
            unpacked_idx += 1
    return packed_tensor

In [4]:
unpacked_tensor = torch.tensor([1, 0, 3, 2], dtype=torch.uint8)

In [5]:
pack_weights(unpacked_tensor, 2)

tensor([177], dtype=torch.uint8)

In [65]:
unpacked_tensor = torch.tensor([1, 0, 3, 2, 3, 3, 3, 3], dtype=torch.uint8)

In [66]:
pack_weights(unpacked_tensor, 2)

tensor([177, 255], dtype=torch.uint8)

In [86]:
def pack_weights_2(uint8tensor, bits):
    if uint8tensor.shape[0] * bits % 8 != 0:
        raise ValueError(f"The input shape needs to be a mutiple of {8 / bits} - got {uint8tensor.shape[0]}")

    num_values = uint8tensor.shape[0] * bits // 8

    num_steps = 8 // bits

    unpacked_idx = 0

    packed_tensor = torch.zeros((num_values), dtype=torch.uint8)

    for i in range(num_values):
        for j in range(num_steps):
            packed_tensor[i] = (packed_tensor[i] << bits) + uint8tensor[unpacked_idx]
            unpacked_idx += 1

    return packed_tensor


def unpack_weights_2(uint8tensor, bits):
    print(f"{uint8tensor.shape[0]=}")

    num_steps = 8 // bits

    print(f"{num_steps=}")

    unpacked_tensor = torch.zeros((uint8tensor.shape[0] * num_steps), dtype=torch.uint8)
    print(f"{unpacked_tensor.shape=}")

    unpacked_idx = 0

    for i in range(uint8tensor.shape[0]):
        for j in range(num_steps):
            print(f"{unpacked_idx=}")
            unpacked_tensor[unpacked_idx] = (uint8tensor[i] >> (bits * j))
            unpacked_idx += 1

    mask = (1 << bits) - 1
    unpacked_tensor &= mask
    return unpacked_tensor

In [87]:
print(unpacked_tensor)

packed_tensor = pack_weights_2(unpacked_tensor, 2)
print(packed_tensor)

unpacked_packed_tensor = unpack_weights_2(packed_tensor, 2)
print(unpacked_packed_tensor)

assert torch.all(unpacked_packed_tensor == unpacked_tensor)

tensor([1, 0, 3, 2, 3, 3, 3, 3], dtype=torch.uint8)
tensor([ 78, 255], dtype=torch.uint8)
uint8tensor.shape[0]=2
num_steps=4
unpacked_tensor.shape=torch.Size([8])
unpacked_idx=0
unpacked_idx=1
unpacked_idx=2
unpacked_idx=3
unpacked_idx=4
unpacked_idx=5
unpacked_idx=6
unpacked_idx=7
tensor([2, 3, 0, 1, 3, 3, 3, 3], dtype=torch.uint8)


AssertionError: 

In [81]:
def pack_weights_3(intweights: torch.Tensor, bits: int) -> torch.Tensor:
    """
    Pack int4 / int2 weights in a uint8 tensor

    What packing means? Assume we have 4 values that are in 2bit but encoded in 8bit
    (because torch does not have native support for 2-bit datatypes)

    > 0000 0011 | 0000 0010 | 0000 0001 | 0000 0000

    We can pack them in a single 8-bit uint value

    > 1110 0100

    Therefore instead of saving 4 values in 8-bit precision we save a single value of 8-bit precision saving 24 bits in total.

    Args:
        intweights (`torch.Tensor`):
            The un-packed `torch.uint8` tensor
        bits (`int`):
            The actual `bits` - can be 2, 4
    """
    original_shape = intweights.shape
    values_per_item = 8 // bits
    row_dim = (original_shape[0] + values_per_item - 1) // values_per_item
    print(f"{original_shape[0]=}")
    print(f"{values_per_item=}")
    print(f"{row_dim=}")

    if len(original_shape) == 1:
        packed_tensor_shape = (row_dim,)
    else:
        packed_tensor_shape = (row_dim, *original_shape[1:])

    packed = torch.zeros(packed_tensor_shape, device=intweights.device, dtype=torch.uint8)
    unpacked = intweights.to(torch.uint8)

    def lshift(t: torch.Tensor, bits: int):
        if t.device.type == "mps":
            # lshift is not supported on MPS device
            return t * (2**bits)
        return t << bits

    print("---")
    it = min(values_per_item, (original_shape[0] // row_dim) + 1)
    for i in range(it):
        start = i * row_dim
        end = min(start + row_dim, original_shape[0])
        print(f"{start=}, {end=}")
        print(f"{unpacked[start:end]=}")
        print(bits * i)
        print(f"{lshift(unpacked[start:end], bits * i)=}")
        print(f"{packed[: (end - start)]=}")
        packed[: (end - start)] |= lshift(unpacked[start:end], bits * i)
        print("---")

    return packed


print(unpacked_tensor)
pack_weights_3(unpacked_tensor, 2)

tensor([1, 0, 3, 2, 3, 3, 3, 3], dtype=torch.uint8)
original_shape[0]=8
values_per_item=4
row_dim=2
---
start=0, end=2
unpacked[start:end]=tensor([1, 0], dtype=torch.uint8)
0
lshift(unpacked[start:end], bits * i)=tensor([1, 0], dtype=torch.uint8)
packed[: (end - start)]=tensor([0, 0], dtype=torch.uint8)
---
start=2, end=4
unpacked[start:end]=tensor([3, 2], dtype=torch.uint8)
2
lshift(unpacked[start:end], bits * i)=tensor([12,  8], dtype=torch.uint8)
packed[: (end - start)]=tensor([1, 0], dtype=torch.uint8)
---
start=4, end=6
unpacked[start:end]=tensor([3, 3], dtype=torch.uint8)
4
lshift(unpacked[start:end], bits * i)=tensor([48, 48], dtype=torch.uint8)
packed[: (end - start)]=tensor([13,  8], dtype=torch.uint8)
---
start=6, end=8
unpacked[start:end]=tensor([3, 3], dtype=torch.uint8)
6
lshift(unpacked[start:end], bits * i)=tensor([192, 192], dtype=torch.uint8)
packed[: (end - start)]=tensor([61, 56], dtype=torch.uint8)
---


tensor([253, 248], dtype=torch.uint8)

In [74]:
pack_weights_3(torch.randint(0, 3, (8, 8)), 2)

original_shape[0]=8
values_per_item=4
row_dim=2
start=0, end=2
unpacked[start:end]=tensor([[1, 1, 0, 0, 0, 0, 0, 1],
        [1, 0, 0, 0, 1, 1, 2, 0]], dtype=torch.uint8)
start=2, end=4
unpacked[start:end]=tensor([[2, 2, 1, 2, 0, 1, 0, 2],
        [0, 0, 0, 2, 0, 0, 2, 1]], dtype=torch.uint8)
start=4, end=6
unpacked[start:end]=tensor([[2, 2, 2, 1, 0, 2, 1, 0],
        [0, 0, 2, 2, 0, 0, 1, 2]], dtype=torch.uint8)
start=6, end=8
unpacked[start:end]=tensor([[1, 2, 2, 0, 0, 0, 1, 0],
        [0, 2, 1, 1, 2, 0, 0, 1]], dtype=torch.uint8)


tensor([[105, 169, 164,  24,   0,  36,  80,   9],
        [  1, 128,  96, 104, 129,   1,  26, 100]], dtype=torch.uint8)

In [69]:
assert (pack_weights(unpacked_tensor, 2) == pack_weights_2(unpacked_tensor, 2)).all()

AssertionError: 

In [13]:
large_unpacked_tensor = torch.randint(0, 3, (1024, ), dtype=torch.uint8)
large_unpacked_tensor

tensor([2, 1, 0,  ..., 1, 0, 0], dtype=torch.uint8)

In [63]:
assert (pack_weights(unpacked_tensor, 2) == pack_weights_2(unpacked_tensor, 2)).all()

pack_weights_2(large_unpacked_tensor, 2)

AssertionError: 

In [15]:
%timeit pack_weights(large_unpacked_tensor, 2)
%timeit pack_weights_2(large_unpacked_tensor, 2)

5.16 ms ± 60.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.13 ms ± 27.1 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
